# Objectification Detector — Final Training and Evaluation

This notebook **trains the final model and evaluates it in the same session**, so every reported number
comes from one clean run on frames the model never saw during training.

**Why it exists:** the earlier metrics notebook rebuilt the train/test split in a new Colab session.
Colab lists files in a different order in each session, so the same `random_state=42` produced a different
test set, and part of it had been used for training. Here the frame list is **sorted** (same order in every
session), the split is **saved to CSV**, and all metrics are computed straight after training.

**Method (same as the project report):** MobileNetV2 (ImageNet) → GlobalAveragePooling → Dense 256 (ReLU) →
Dropout 0.5 → Dense 1 (sigmoid); 224×224 RGB frames scaled to [0, 1]; stratified 80/20 train/test split
(and 20% of training for validation); stage 1: base frozen, Adam 1e-3, 20 epochs; stage 2: last 20 layers
unfrozen, Adam 1e-5, 10 epochs.

**How to run**
1. *Runtime → Change runtime type → **T4 GPU*** → Save.
2. *Runtime → **Run all***. Allow Google Drive access when asked (it reads `frames_cleaned.zip` from your Drive;
   if it is not there, an upload button appears).
3. Wait about 30–35 minutes. At the end **`final_results.zip`** downloads (a copy is also saved to your Drive
   in the folder `objectification_final_results`).

In [ ]:
# ── Settings ──────────────────────────────────────────────────────────────────
DRIVE_ZIP = "/content/drive/MyDrive/frames_cleaned.zip"   # where frames_cleaned.zip is on Google Drive
SEED = 42
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS_HEAD = 20        # stage 1: MobileNetV2 base frozen
EPOCHS_FINE = 10        # stage 2: last 20 base layers unfrozen
LR_HEAD = 1e-3
LR_FINE = 1e-5
UNFREEZE_LAST = 20
THRESHOLD = 0.5
LIME_SAMPLES = 500
RESULTS_DIR = "/content/final_results"
RUN_UNSEEN_ADS_CHECK = False   # optional extra (~25 min more): also train/test with whole ads held out

In [ ]:
# ── Set-up ────────────────────────────────────────────────────────────────────
import os, sys, json, time, random, shutil, zipfile, subprocess
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, confusion_matrix, accuracy_score, roc_curve,
                             roc_auc_score, precision_recall_curve, average_precision_score)

IN_COLAB = "google.colab" in sys.modules
tf.keras.utils.set_random_seed(SEED)          # seeds Python, NumPy and TensorFlow
os.makedirs(RESULTS_DIR, exist_ok=True)
T0 = time.time()

gpus = tf.config.list_physical_devices("GPU")
print("TensorFlow", tf.__version__, "| Keras", tf.keras.__version__ if hasattr(tf.keras, "__version__") else "")
print("GPU:", gpus if gpus else "NONE — training will be very slow. Use Runtime > Change runtime type > T4 GPU.")

In [ ]:
# ── Get the frames (frames_cleaned.zip) ───────────────────────────────────────
def find_data_root(base):
    # The folder that contains both OBJ and NON_OBJ.
    for root, dirs, _ in os.walk(base):
        if "OBJ" in dirs and "NON_OBJ" in dirs:
            return root
    raise FileNotFoundError(f"No folder with both OBJ and NON_OBJ inside {base}")

if IN_COLAB:
    from google.colab import drive, files
    drive.mount("/content/drive")
    candidates = [DRIVE_ZIP, "/content/drive/My Drive/frames_cleaned.zip"]
    zip_src = next((p for p in candidates if os.path.exists(p)), None)
    if zip_src:
        shutil.copy(zip_src, "/content/frames_cleaned.zip")
        print("Copied", zip_src)
    else:
        print("frames_cleaned.zip is not on Drive at", DRIVE_ZIP, "- please upload it now:")
        up = files.upload()
        name = next(iter(up))
        if name != "frames_cleaned.zip":
            shutil.move(name, "/content/frames_cleaned.zip")
    if not os.path.isdir("/content/frames_cleaned"):
        with zipfile.ZipFile("/content/frames_cleaned.zip") as z:
            z.extractall("/content/frames_cleaned")
    DATA_ROOT = find_data_root("/content/frames_cleaned")

print("Data folder:", DATA_ROOT)

In [ ]:
# ── List every frame in a FIXED (sorted) order ────────────────────────────────
rows = []
for label_str, label in (("NON_OBJ", 0), ("OBJ", 1)):
    class_dir = os.path.join(DATA_ROOT, label_str)
    for video in sorted(os.listdir(class_dir)):            # sorted: identical order in every session
        vdir = os.path.join(class_dir, video)
        if not os.path.isdir(vdir):
            continue
        for f in sorted(os.listdir(vdir)):
            if f.lower().endswith((".jpg", ".jpeg", ".png")):
                rows.append({"frame_path": os.path.join(vdir, f), "label": label,
                             "label_str": label_str, "video": f"{label_str}/{video}"})
df = pd.DataFrame(rows)

videos = df.groupby("label_str")["video"].nunique()
frames = df["label_str"].value_counts()
print(f"Ads (video folders): {df['video'].nunique()}  ->  OBJ {videos.get('OBJ', 0)}, NON_OBJ {videos.get('NON_OBJ', 0)}")
print(f"Frames: {len(df)}  ->  OBJ {frames.get('OBJ', 0)}, NON_OBJ {frames.get('NON_OBJ', 0)}")

In [ ]:
# ── Split: stratified 80/20 train/test, then 20% of train for validation ─────
train_full, test_df = train_test_split(df, test_size=0.2, random_state=SEED, stratify=df["label"])
train_df, val_df = train_test_split(train_full, test_size=0.2, random_state=SEED, stratify=train_full["label"])
for name, part in (("train", train_df), ("val", val_df), ("test", test_df)):
    part.to_csv(os.path.join(RESULTS_DIR, f"split_{name}.csv"), index=False)   # the exact split, saved
    print(f"{name:5s}: {len(part):5d} frames  (OBJ {int(part['label'].sum())}, NON_OBJ {int((part['label'] == 0).sum())})")

In [ ]:
# ── Data generators (pixels scaled to [0, 1]; augmentation on training only) ──
from tensorflow.keras.preprocessing.image import ImageDataGenerator

def make_generators(train_df, val_df, test_df):
    aug = ImageDataGenerator(rescale=1./255, rotation_range=20, width_shift_range=0.2, height_shift_range=0.2,
                             shear_range=0.2, zoom_range=0.2, horizontal_flip=True)
    plain = ImageDataGenerator(rescale=1./255)
    common = dict(x_col="frame_path", y_col="label_str", target_size=IMG_SIZE, batch_size=BATCH_SIZE,
                  class_mode="binary", classes=["NON_OBJ", "OBJ"])   # NON_OBJ = 0, OBJ = 1
    tr = aug.flow_from_dataframe(train_df, shuffle=True, seed=SEED, **common)
    va = plain.flow_from_dataframe(val_df, shuffle=False, **common)
    te = plain.flow_from_dataframe(test_df, shuffle=False, **common)   # fixed order for predictions
    return tr, va, te

train_gen, val_gen, test_gen = make_generators(train_df, val_df, test_df)
print("Class indices:", train_gen.class_indices)

In [ ]:
# ── Model: MobileNetV2 (ImageNet) + small classifier head ─────────────────────
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.models import Model

def build_model():
    base = MobileNetV2(weights="imagenet", include_top=False, input_shape=IMG_SIZE + (3,))
    base.trainable = False
    x = GlobalAveragePooling2D()(base.output)
    x = Dense(256, activation="relu")(x)
    x = Dropout(0.5)(x)
    out = Dense(1, activation="sigmoid")(x)
    model = Model(inputs=base.input, outputs=out)
    model.compile(optimizer=tf.keras.optimizers.Adam(LR_HEAD), loss="binary_crossentropy", metrics=["accuracy"])
    return model, base

def fine_tune_setup(model, base):
    for layer in base.layers[-UNFREEZE_LAST:]:
        layer.trainable = True
    model.compile(optimizer=tf.keras.optimizers.Adam(LR_FINE), loss="binary_crossentropy", metrics=["accuracy"])

model, base = build_model()
print(f"Total parameters: {model.count_params():,}")

In [ ]:
# ── Stage 1: train the head (base frozen) ─────────────────────────────────────
t = time.time()
hist1 = model.fit(train_gen, epochs=EPOCHS_HEAD, validation_data=val_gen)
stage1_loss, stage1_acc = model.evaluate(test_gen, verbose=0)
print(f"Stage 1 done in {(time.time() - t) / 60:.1f} min. Test accuracy before fine-tuning: {stage1_acc * 100:.2f}%")

In [ ]:
# ── Stage 2: fine-tune the last 20 layers ─────────────────────────────────────
t = time.time()
fine_tune_setup(model, base)
hist2 = model.fit(train_gen, epochs=EPOCHS_FINE, validation_data=val_gen)
final_loss, final_acc = model.evaluate(test_gen, verbose=0)
print(f"Stage 2 done in {(time.time() - t) / 60:.1f} min. Final test accuracy: {final_acc * 100:.2f}%")

# Save the model straight away (the dashboard backend loads objectification_model.h5)
model.save(os.path.join(RESULTS_DIR, "objectification_model.h5"))
model.save(os.path.join(RESULTS_DIR, "objectification_model.keras"))
history = pd.concat([pd.DataFrame(hist1.history).assign(stage=1), pd.DataFrame(hist2.history).assign(stage=2)],
                    ignore_index=True)
history.index = history.index + 1
history.index.name = "epoch"
history.to_csv(os.path.join(RESULTS_DIR, "training_history.csv"))
print("Model and training history saved.")

In [ ]:
# ── Full evaluation on the held-out test frames ───────────────────────────────
probs = model.predict(test_gen, verbose=0).ravel()          # P(OBJ) for each test frame, in test_gen order
y_true = np.asarray(test_gen.classes)                        # same order (shuffle=False)
y_pred = (probs >= THRESHOLD).astype(int)

report = classification_report(y_true, y_pred, target_names=["NON_OBJ", "OBJ"], digits=4, output_dict=True)
cm = confusion_matrix(y_true, y_pred)
tn, fp, fn, tp = (int(v) for v in cm.ravel())
auc = roc_auc_score(y_true, probs)
ap = average_precision_score(y_true, probs)
majority = max(np.mean(y_true), 1 - np.mean(y_true))

pd.DataFrame({"frame_path": test_gen.filepaths, "true": y_true, "prob_obj": probs, "pred": y_pred}).to_csv(
    os.path.join(RESULTS_DIR, "test_predictions.csv"), index=False)

metrics = {
    "dataset": {"ads": int(df["video"].nunique()), "ads_obj": int(videos.get("OBJ", 0)),
                "ads_non_obj": int(videos.get("NON_OBJ", 0)), "frames": int(len(df)),
                "frames_obj": int(frames.get("OBJ", 0)), "frames_non_obj": int(frames.get("NON_OBJ", 0)),
                "train": int(len(train_df)), "val": int(len(val_df)), "test": int(len(test_df)),
                "test_obj": int(np.sum(y_true == 1)), "test_non_obj": int(np.sum(y_true == 0))},
    "test_accuracy_before_fine_tuning": round(float(stage1_acc), 4),
    "test_accuracy": round(float(accuracy_score(y_true, y_pred)), 4),
    "test_loss": round(float(final_loss), 4),
    "majority_class_baseline": round(float(majority), 4),
    "roc_auc": round(float(auc), 4),
    "average_precision": round(float(ap), 4),
    "confusion_matrix": {"tn": tn, "fp": fp, "fn": fn, "tp": tp},
    "per_class": {k: {m: round(float(v), 4) for m, v in report[k].items()} for k in ("NON_OBJ", "OBJ")},
    "macro_avg": {m: round(float(v), 4) for m, v in report["macro avg"].items()},
    "weighted_avg": {m: round(float(v), 4) for m, v in report["weighted avg"].items()},
    "final_epoch": {k: round(float(v), 4) for k, v in history.iloc[-1].items() if k != "stage"},
    "settings": {"seed": SEED, "img_size": list(IMG_SIZE), "batch_size": BATCH_SIZE, "epochs_head": EPOCHS_HEAD,
                 "epochs_fine": EPOCHS_FINE, "lr_head": LR_HEAD, "lr_fine": LR_FINE,
                 "unfreeze_last": UNFREEZE_LAST, "threshold": THRESHOLD},
    "versions": {"tensorflow": tf.__version__},
}
with open(os.path.join(RESULTS_DIR, "metrics.json"), "w") as f:
    json.dump(metrics, f, indent=2)

print(classification_report(y_true, y_pred, target_names=["NON_OBJ", "OBJ"], digits=4))
print(f"Accuracy {metrics['test_accuracy'] * 100:.2f}%  |  before fine-tuning {stage1_acc * 100:.2f}%  |  "
      f"ROC-AUC {auc:.4f}  |  AP {ap:.4f}  |  majority-class baseline {majority * 100:.2f}%")
print(f"Confusion matrix: TN {tn}  FP {fp}  FN {fn}  TP {tp}")

In [ ]:
# ── Figures: training curves, confusion matrix, ROC, precision-recall ─────────
def save(fig, name):
    fig.savefig(os.path.join(RESULTS_DIR, name), dpi=160, bbox_inches="tight")
    plt.show()

ep = history.index.values
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
for ax, (tr, va, title) in zip(axes, (("accuracy", "val_accuracy", "Accuracy"), ("loss", "val_loss", "Loss"))):
    ax.plot(ep, history[tr], marker="o", ms=3, label="training")
    ax.plot(ep, history[va], marker="o", ms=3, label="validation")
    ax.axvline(EPOCHS_HEAD + 0.5, color="gray", ls="--", lw=1)
    ax.text(EPOCHS_HEAD + 0.7, ax.get_ylim()[0], "fine-tuning starts", color="gray", fontsize=8, va="bottom")
    ax.set_xlabel("Epoch"); ax.set_title(title); ax.grid(alpha=0.3); ax.legend()
save(fig, "training_curves.png")

fig, ax = plt.subplots(figsize=(4.8, 4.2))
ax.imshow(cm, cmap="Blues")
for (i, j), v in np.ndenumerate(cm):
    ax.text(j, i, str(v), ha="center", va="center", fontsize=14, color="white" if v > cm.max() / 2 else "black")
ax.set_xticks([0, 1], ["NON_OBJ", "OBJ"]); ax.set_yticks([0, 1], ["NON_OBJ", "OBJ"])
ax.set_xlabel("Predicted"); ax.set_ylabel("Actual"); ax.set_title(f"Confusion matrix ({len(y_true)} test frames)")
save(fig, "confusion_matrix.png")

fpr, tpr, _ = roc_curve(y_true, probs)
fig, ax = plt.subplots(figsize=(4.8, 4.2))
ax.plot(fpr, tpr, lw=2, label=f"AUC = {auc:.2f}"); ax.plot([0, 1], [0, 1], "--", color="gray")
ax.set_xlabel("False positive rate"); ax.set_ylabel("True positive rate"); ax.set_title("ROC curve")
ax.grid(alpha=0.3); ax.legend(loc="lower right")
save(fig, "roc_curve.png")

prec, rec, _ = precision_recall_curve(y_true, probs)
fig, ax = plt.subplots(figsize=(4.8, 4.2))
ax.plot(rec, prec, lw=2, label=f"Average precision = {ap:.2f}")
ax.set_xlabel("Recall"); ax.set_ylabel("Precision"); ax.set_title("Precision-recall curve")
ax.grid(alpha=0.3); ax.legend(loc="lower left")
save(fig, "pr_curve.png")

In [ ]:
# ── Explainability: Grad-CAM and LIME on held-out test frames ─────────────────
try:
    try:
        from lime import lime_image
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "lime"], check=True)
        from lime import lime_image
    from skimage.segmentation import mark_boundaries

    final_layer = model.layers[-1]
    grad_model = tf.keras.Model(model.inputs, [model.get_layer("Conv_1").output, final_layer.input])
    W, B = (tf.constant(w, dtype=tf.float32) for w in final_layer.get_weights())

    def gradcam(img01):
        # Regions that push the score towards OBJ. Uses the logit (before the sigmoid) so it never vanishes.
        with tf.GradientTape() as tape:
            conv, h = grad_model(img01[None].astype("float32"), training=False)
            target = (tf.matmul(h, W) + B)[:, 0]     # computed inside the tape, so it is recorded
        grads = tape.gradient(target, conv)
        cam = tf.nn.relu(tf.reduce_sum(conv * tf.reduce_mean(grads, axis=(1, 2))[:, None, None, :], axis=-1))[0].numpy()
        return cam / cam.max() if cam.max() > 0 else cam

    def load01(path):
        img = tf.keras.utils.load_img(path, target_size=IMG_SIZE)
        return tf.keras.utils.img_to_array(img) / 255.0             # [0, 1] — scaled ONCE

    def predict_fn(batch):
        p = model.predict(np.asarray(batch, dtype="float32"), verbose=0)
        return np.hstack([1 - p, p])                                 # [P(NON_OBJ), P(OBJ)]

    paths = np.array(test_gen.filepaths)
    order = np.argsort(-probs)

    def not_blank(i):
        # Skip black / near-empty frames: they make useless examples in a figure.
        return float(np.std(load01(paths[i]))) > 0.08

    picks = []
    tp_idx = [i for i in order if y_true[i] == 1 and y_pred[i] == 1]
    tn_idx = [i for i in order[::-1] if y_true[i] == 0 and y_pred[i] == 0]
    fp_idx = [i for i in order if y_true[i] == 0 and y_pred[i] == 1]
    fn_idx = [i for i in order[::-1] if y_true[i] == 1 and y_pred[i] == 0]
    for group, n in ((tp_idx, 3), (tn_idx, 1), (fp_idx, 1), (fn_idx, 1)):
        chosen = []
        for i in group[:60]:
            if len(chosen) == n:
                break
            if not_blank(i):
                chosen.append(i)
        picks += chosen

    explainer = lime_image.LimeImageExplainer(random_state=SEED)
    fig, axes = plt.subplots(len(picks), 3, figsize=(10, 3.1 * len(picks)))
    axes = np.atleast_2d(axes)
    for r, i in enumerate(picks):
        img = load01(paths[i])
        cam = tf.image.resize(gradcam(img)[..., None], IMG_SIZE).numpy()[..., 0]
        exp = explainer.explain_instance(img.astype("double"), predict_fn, top_labels=2, hide_color=0,
                                         num_samples=LIME_SAMPLES, batch_size=50)
        temp, mask = exp.get_image_and_mask(1, positive_only=True, num_features=5, hide_rest=False)
        kind = {(1, 1): "TP", (0, 0): "TN", (0, 1): "FP", (1, 0): "FN"}[(int(y_true[i]), int(y_pred[i]))]
        axes[r, 0].imshow(img); axes[r, 0].set_title(f"{kind}: true {'OBJ' if y_true[i] else 'NON_OBJ'}, P(OBJ)={probs[i]:.2f}", fontsize=9)
        axes[r, 1].imshow(img); axes[r, 1].imshow(cam, cmap="jet", alpha=0.45); axes[r, 1].set_title("Grad-CAM (red = towards OBJ)", fontsize=9)
        axes[r, 2].imshow(mark_boundaries(temp, mask)); axes[r, 2].set_title("LIME (outlined = supports OBJ)", fontsize=9)  # temp is already 0-1
        for a in axes[r]:
            a.axis("off")
    fig.tight_layout()
    save(fig, "gradcam_lime.png")
except Exception as e:   # never lose the main results because of a figure
    print("Explainability figure skipped:", repr(e))

In [ ]:
# ── OPTIONAL: whole ads held out (does it work on ads it has never seen?) ─────
# Off by default (adds ~25 minutes). Set RUN_UNSEEN_ADS_CHECK = True in the first cell to run it.
if RUN_UNSEEN_ADS_CHECK:
    from sklearn.model_selection import GroupShuffleSplit
    gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
    tr_i, te_i = next(gss.split(df, df["label"], groups=df["video"]))
    g_train_full, g_test = df.iloc[tr_i], df.iloc[te_i]
    gss2 = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
    tr2, va2 = next(gss2.split(g_train_full, g_train_full["label"], groups=g_train_full["video"]))
    g_tr, g_va = g_train_full.iloc[tr2], g_train_full.iloc[va2]
    gtr, gva, gte = make_generators(g_tr, g_va, g_test)
    m2, b2 = build_model()
    m2.fit(gtr, epochs=EPOCHS_HEAD, validation_data=gva)
    fine_tune_setup(m2, b2)
    m2.fit(gtr, epochs=EPOCHS_FINE, validation_data=gva)
    p2 = m2.predict(gte, verbose=0).ravel()
    t2 = np.asarray(gte.classes)
    per_ad = pd.DataFrame({"video": g_test["video"].values, "label": t2, "prob": p2}).groupby("video").agg(
        label=("label", "first"), mean_prob=("prob", "mean"))
    per_ad["verdict"] = (per_ad["mean_prob"] >= THRESHOLD).astype(int)
    unseen = {"test_ads": int(len(per_ad)), "test_frames": int(len(t2)),
              "frame_accuracy": round(float(np.mean((p2 >= THRESHOLD).astype(int) == t2)), 4),
              "ad_level_accuracy": round(float(np.mean(per_ad["verdict"] == per_ad["label"])), 4)}
    with open(os.path.join(RESULTS_DIR, "unseen_ads_metrics.json"), "w") as f:
        json.dump(unseen, f, indent=2)
    print("Ads never seen in training:", unseen)
else:
    print("Optional unseen-ads check skipped (RUN_UNSEEN_ADS_CHECK = False).")

In [ ]:
# ── Package everything and download ───────────────────────────────────────────
print(json.dumps({k: metrics[k] for k in ("dataset", "test_accuracy_before_fine_tuning", "test_accuracy",
                                          "roc_auc", "average_precision", "confusion_matrix")}, indent=2))
zip_base = RESULTS_DIR.rstrip("/")
shutil.make_archive(zip_base, "zip", RESULTS_DIR)
print("Created", zip_base + ".zip", f"({os.path.getsize(zip_base + '.zip') / 1e6:.1f} MB) in {(time.time() - T0) / 60:.1f} min total")
if IN_COLAB:
    backup = "/content/drive/MyDrive/objectification_final_results"
    os.makedirs(backup, exist_ok=True)
    shutil.copy(zip_base + ".zip", backup)
    print("Backup copy saved to Google Drive:", backup)
    from google.colab import files
    files.download(zip_base + ".zip")